# Wan 2.1 x FastAPI — Mode CPU

> **Runtime CPU** — Pas besoin de GPU !
>
> Le rendu sera plus lent (quelques minutes par segment) mais entierement fonctionnel.
>
> Ajoutez vos secrets dans l'icone cle a gauche :
> - `NGROK_TOKEN` — votre token ngrok (https://dashboard.ngrok.com)


In [ ]:
# Cellule 1 : Installation des dependances
!pip install -q diffusers transformers accelerate sentencepiece
!pip install -q fastapi 'uvicorn[standard]' pyngrok pillow numpy
print('Dependances installees')

In [ ]:
# Cellule 2 : Configuration ngrok
from google.colab import userdata
import os

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(8000, 'http')
NGROK_URL = tunnel.public_url
os.environ['NGROK_URL'] = NGROK_URL

print('Tunnel actif !')
print('=' * 50)
print(f'NEXT_PUBLIC_COLAB_URL={NGROK_URL}')
print('=' * 50)
print('Copiez cette URL dans votre .env.local')

In [ ]:
# Cellule 3 : Telechargement du serveur FastAPI
import os
if not os.path.exists('wan21_server.py'):
    print('Telechargement depuis GitHub...')
    !wget -q https://raw.githubusercontent.com/aivideonew36-cyber/ai-video-generator/main/colab/wan21_server.py
print('Serveur pret')

In [ ]:
# Cellule 4 : Demarrage FastAPI + chargement Wan 2.1 sur CPU
# Le chargement du modele prend 3-5 minutes sur CPU
import subprocess, time, threading

def run_server():
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'wan21_server:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end='')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(15)
print(f'Serveur demarre ! Health : {NGROK_URL}/health')

In [ ]:
# Cellule 5 : Test /health
import requests
r = requests.get(f'{NGROK_URL}/health', timeout=15)
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False))